# Stage 2 — MIMIC-IV feature engineering

Run this notebook only in your credentialed Kaggle/Colab environment. Patient-level MIMIC-IV data and all derived Parquet/NPZ files must remain in that environment under the gitignored `data/` directories. Do not display patient rows, upload them to an API, or commit them.

This stage builds both required representations from the first 6 ICU hours: a static table for classic models/MLP and an hourly value tensor plus an aligned observation mask for the LSTM. Median imputation and scaling are deliberately postponed until Stage 3 so they can be fitted on training data only.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from google.cloud import bigquery

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.config import DATA_INTERIM, DATA_PROCESSED, N_HOURS, RESULTS_FIGURES
from src.features import (
    DYNAMIC_FEATURES,
    VASOACTIVE_FEATURES,
    estimate_feature_query_bytes,
    run_stage2,
)

## Confirm the Stage 1 input

Do not continue until the protected 5-positive/5-negative label audit from Stage 1 has been completed. This cell checks only aggregate properties and never displays identifiers.

In [ ]:
cohort_path = DATA_PROCESSED / "cohort_mimiciv.parquet"
if not cohort_path.is_file():
    raise FileNotFoundError(
        f"Missing {cohort_path}. Run notebooks/01_cohort.ipynb first."
    )

cohort_check = pd.read_parquet(
    cohort_path, columns=["subject_id", "stay_id", "label"]
)
assert cohort_check["stay_id"].is_unique
assert set(cohort_check["label"].unique()).issubset({0, 1})
assert cohort_check["label"].nunique() == 2
print(f"Cohort stays: {len(cohort_check):,}")
print(f"Positive prevalence: {cohort_check['label'].mean():.1%}")
del cohort_check

## Connect with your own billing project

Use the Google Cloud project linked to your credentialed PhysioNet account. Authentication is intentionally not stored in this repository. In Colab, authenticate through the standard Google authentication UI before running this cell.

In [ ]:
billing_project = "GOOGLE_CLOUD_PROJECT_ID"
if billing_project == "GOOGLE_CLOUD_PROJECT_ID":
    raise ValueError("Replace GOOGLE_CLOUD_PROJECT_ID with your billing project.")
client = bigquery.Client(project=billing_project)

## Dry run

Estimate the scan size before running the protected feature query. The query embeds exactly the same parameterized cohort definition used in Stage 1.

In [ ]:
feature_bytes = estimate_feature_query_bytes(client)
print(f"Feature query estimate: {feature_bytes / 2**30:.3f} GiB")

## Execute Stage 2

This runs the feature query, validates the returned event metadata and time boundaries, handles fixed outlier ranges, and writes all protected artifacts. Only aggregate counts, shapes, and paths are printed.

Urine output is summed within each hour. Vasoactive rates are duration-weighted within each hour; no documented infusion is represented as zero, and medication channels are never forward-filled.

In [ ]:
artifacts = run_stage2(client, cohort_path=cohort_path)
print(f"Validated event rows: {artifacts.event_count:,}")
print(f"Static matrix shape: {artifacts.static_shape}")
print(f"Hourly value/mask shape: {artifacts.hourly_shape}")
print(f"Protected events: {artifacts.events_path}")
print(f"Protected static features: {artifacts.static_path}")
print(f"Protected hourly tensor: {artifacts.hourly_path}")
print(f"Aggregate feature dictionary: {artifacts.dictionary_path}")
print(f"Aggregate summary statistics: {artifacts.summary_path}")

## Aggregate acceptance checks

These checks verify patient alignment, the fixed `[patients, 6, 29]` channel structure, binary masks, and the absence of future-label columns. They do not expose patient-level values.

In [ ]:
static_check = pd.read_parquet(
    artifacts.static_path, columns=["subject_id", "stay_id", "hadm_id", "label"]
)
with np.load(artifacts.hourly_path, allow_pickle=False) as hourly:
    values_shape = hourly["values"].shape
    mask_shape = hourly["mask"].shape
    saved_features = tuple(hourly["feature_names"].tolist())
    assert values_shape == mask_shape
    assert values_shape == (len(static_check), N_HOURS, len(DYNAMIC_FEATURES))
    assert saved_features == DYNAMIC_FEATURES
    assert np.array_equal(hourly["stay_ids"], static_check["stay_id"].to_numpy())
    assert np.array_equal(hourly["subject_ids"], static_check["subject_id"].to_numpy())
    assert np.array_equal(hourly["labels"], static_check["label"].to_numpy())
    assert np.isin(hourly["mask"], (0, 1)).all()

assert len(VASOACTIVE_FEATURES) == 7
assert static_check["stay_id"].is_unique
print("Stage 2 structural checks passed.")
del static_check

## Missingness diagnosis and the 6-hour decision

The table and figure below contain only aggregate coverage. Review the key variables before keeping `N=6`. A high missing rate is not an automatic reason to delete a feature: measurement itself can be informative, and the LSTM retains the observation mask. Vasoactive absence is a known zero rather than a missing measurement.

In [ ]:
feature_dictionary = pd.read_csv(artifacts.dictionary_path)
coverage = feature_dictionary.sort_values(
    ["patient_missing_rate", "feature_name"], ascending=[False, True]
)
display(
    coverage[[
        "feature_name",
        "source_table",
        "unit",
        "hourly_aggregation",
        "patients_with_events",
        "patient_missing_rate",
    ]]
)

RESULTS_FIGURES.mkdir(parents=True, exist_ok=True)
missingness_path = RESULTS_FIGURES / "feature_missingness_mimiciv.png"
plot_data = coverage.sort_values("patient_missing_rate")
fig, ax = plt.subplots(figsize=(8, 9))
ax.barh(plot_data["feature_name"], plot_data["patient_missing_rate"] * 100)
ax.axvline(70, color="tab:red", linestyle="--", linewidth=1, label="70% review line")
ax.set_xlabel("Patients without an observation (%)")
ax.set_ylabel("Feature")
ax.set_xlim(0, 100)
ax.legend()
fig.tight_layout()
fig.savefig(missingness_path, dpi=200, bbox_inches="tight")
plt.show()
print(f"Aggregate missingness figure saved to: {missingness_path}")

In [ ]:
key_features = [
    "heart_rate",
    "sbp",
    "resp_rate",
    "temperature",
    "spo2",
    "wbc",
    "creatinine",
    "lactate",
    "bilirubin_total",
    "platelet",
    "gcs",
    "urineoutput",
]
key_coverage = feature_dictionary.loc[
    feature_dictionary["feature_name"].isin(key_features),
    ["feature_name", "observed_patients", "patient_missing_rate"],
].sort_values("patient_missing_rate", ascending=False)
display(key_coverage)
very_sparse = key_coverage.loc[key_coverage["patient_missing_rate"].gt(0.70)]
if very_sparse.empty:
    print("No key feature exceeds the 70% review line; N=6 remains plausible.")
else:
    print(
        f"Review N=6 before Stage 3: {len(very_sparse)} key feature(s) exceed 70% missingness."
    )

## Protected two-stay feature audit

This creates three small protected files for manual comparison: source events, static aggregates, and the six hourly rows with their masks. It deliberately prints neither identifiers nor values. When a vasoactive stay exists, the selection prioritizes one such stay. Open the files only inside the controlled environment and verify at least one regular measurement, urine-output summation, and one vasoactive interval if available.

In [ ]:
vasoactive_stays = pd.read_parquet(
    artifacts.events_path,
    columns=["stay_id"],
    filters=[("feature_name", "in", list(VASOACTIVE_FEATURES))],
).drop_duplicates()
with np.load(artifacts.hourly_path, allow_pickle=False) as hourly:
    rng = np.random.default_rng(42)
    all_stays = hourly["stay_ids"]
    sample_size = min(2, len(all_stays))
    vasoactive_positions = np.flatnonzero(
        np.isin(all_stays, vasoactive_stays["stay_id"].to_numpy())
    )
    selected_positions = []
    if len(vasoactive_positions):
        selected_positions.append(int(rng.choice(vasoactive_positions)))
    remaining_positions = np.setdiff1d(
        np.arange(len(all_stays)), selected_positions
    )
    needed = sample_size - len(selected_positions)
    if needed:
        selected_positions.extend(
            int(value)
            for value in rng.choice(remaining_positions, size=needed, replace=False)
        )
    sample_positions = np.sort(np.asarray(selected_positions, dtype=int))
    selected_stays = [int(value) for value in hourly["stay_ids"][sample_positions]]
    selected_features = tuple(hourly["feature_names"].tolist())
    audit_values = hourly["values"][sample_positions]
    audit_masks = hourly["mask"][sample_positions]

audit_events = pd.read_parquet(
    artifacts.events_path, filters=[("stay_id", "in", selected_stays)]
).sort_values(["stay_id", "hour_bin", "feature_name", "charttime"])
audit_static = pd.read_parquet(
    artifacts.static_path, filters=[("stay_id", "in", selected_stays)]
)
hourly_frames = []
for local_index, stay_id in enumerate(selected_stays):
    frame = pd.DataFrame(audit_values[local_index], columns=selected_features)
    frame.insert(0, "hour_bin", np.arange(N_HOURS))
    frame.insert(0, "stay_id", stay_id)
    for feature_index, feature_name in enumerate(selected_features):
        frame[f"{feature_name}_mask"] = audit_masks[
            local_index, :, feature_index
        ]
    hourly_frames.append(frame)
audit_hourly = pd.concat(hourly_frames, ignore_index=True)

DATA_INTERIM.mkdir(parents=True, exist_ok=True)
audit_events_path = DATA_INTERIM / "feature_audit_events.parquet"
audit_static_path = DATA_INTERIM / "feature_audit_static.parquet"
audit_hourly_path = DATA_INTERIM / "feature_audit_hourly.parquet"
audit_events.to_parquet(audit_events_path, index=False)
audit_static.to_parquet(audit_static_path, index=False)
audit_hourly.to_parquet(audit_hourly_path, index=False)
del audit_events, audit_static, audit_hourly, hourly_frames, vasoactive_stays
print("Protected two-stay audit package created under data/interim/.")

## Stop before preprocessing

Do **not** fit medians or scalers in this notebook using the complete cohort. After reviewing the aggregate missingness and protected audit package, Stage 3 will create patient-grouped train/development/test splits and fit every preprocessing statistic on the relevant training fold only.